In [ ]:
import numpy as np
from pycvxset import Polytope
from adaptive_tools import SetUpdater
from invariance_tools import GainSynthesis
from cartesian_product import cartesian_product
from cRMPC import CRMPC

In [ ]:
A = np.array(
            [
                [
                    [ 1.00000000e+00,  0.00000000e+00,  0.00000000e+00, 0.00000000e+00],
                    [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 0.00000000e+00],
                    [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 0.00000000e+00],
                    [ 6.55847073e-03,  0.00000000e+00,  0.00000000e+00, 0.00000000e+00]],
                [
                    [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 0.00000000e+00],
                    [ 9.62477033e-01, -2.70418272e-01, -3.54535263e-01, -1.08121033e-01],
                    [ 6.03978994e-02, -3.51946037e-01, -6.16284799e-01, 1.82183285e-01],
                    [-2.98716243e-03, -3.30481599e-02, -3.49541770e-05, 1.52896134e-02]],
                [
                    [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 0.00000000e+00],
                    [-4.21460547e-02,  1.48734527e-01,  2.49708275e-01, 1.46475140e-01],
                    [ 9.32949059e-01,  2.33489450e-01,  3.84304411e-01, -1.35325598e-02],
                    [-9.96893529e-03,  1.42242452e-02,  5.02034057e-03, -4.66406071e-02]],
                [
                    [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 0.00000000e+00],
                    [-2.59938961e-02, -2.63172399e-02,  4.65109389e-03, 3.00565144e-01],
                    [-1.59201270e-03, -2.02848177e-02, -2.60380154e-02, 3.90983528e-01],
                    [ 8.70485640e-01, -5.48457771e-02,  3.24459921e-02, 6.67191728e-02]]])
B = np.array(
            [
                [
                    [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 0.00000000e+00],
                    [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 0.00000000e+00]],
                [
                    [ 8.40916986e-02, -6.74379191e-01,  4.14184808e-01, 6.96994653e-02],
                    [ 1.37701713e-02,  2.87919480e-02, -9.12432271e-02, -2.99699931e-01]],
                [
                    [-1.02160995e-02,  5.14789438e-01, -3.00694970e-01, 2.04723249e-01],
                    [ 2.58556998e-04, -4.44445006e-03,  4.10364069e-02, 3.52610402e-01]],
                [
                    [ 2.28217366e-02, -1.76918468e-02,  8.71910634e-02, 6.08327604e-01],
                    [ 1.31207096e-01,  2.19817108e-03,  5.56598339e-02, -2.28968692e-01]]])

C = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
C = np.stack([C, np.zeros((2, 4)), np.zeros((2, 4)), np.zeros((2, 4))], axis=2)

A_B=np.concatenate( (A, B), axis=1).transpose(2, 0, 1)

E = Polytope(
                A=np.vstack((np.eye(2), -np.eye(2))),
                b=np.concatenate((np.ones(2) * 0.01, np.ones(2) * 0.01)),
            )
W = Polytope(
                A=np.block([[np.eye(4)], [-np.eye(4)]]), b=0.05 * np.ones((8, 1))
            )
Theta_c = Polytope(V=np.eye(0))

In [ ]:
B.shape, A.shape, C.shape

In [ ]:
max_p, min_p = ([np.float64(-0.23350), np.float64(1.37460), np.float64(0.31949)], [np.float64(-1.81833),np.float64(-0.26129),np.float64(-0.61740)])

Theta = Polytope(A=np.block([[np.eye(3)], [-np.eye(3)]]), b=np.block([np.array(max_p), -np.array(min_p)]))  # np.block([np.array(max_p), -np.array(min_p)])
Theta.plot()
Theta.V

SetUpdater(A_B, C.transpose(2, 0, 1), Theta, Theta_c, W, E, 1)

In [ ]:
lbx, ubx = (np.array([-1e4,-0.46, -0.01, -1.90]), np.array([1e4, 0.46, 0.01, 1.90]))
x = Polytope(A=np.block([[np.eye(4)], [-np.eye(4)]]), b=np.concatenate(([1e4, 0.46, 0.01, 1.90], np.array([1e4, 0.46, 0.01, 1.90]))))
u = Polytope(A=np.block([[np.eye(2)], [-np.eye(2)]]), b=np.concatenate(([0.46, 1.90], np.array([0.46, 1.90]))))
z = cartesian_product(x, u)
Q = np.eye(4)
R = np.diag([1/(0.46**2), 1/(1.90**2)])  # np.diag([1/(0.46**2), 1/(1.90**2), 1, 1])
gain_synth = GainSynthesis(
            A, B,
            Q=Q, R=R,
            Z_bnd=z, W=W,
            mode='LQR', contraction_factor=1)
gain_synth.synthesize_controller()

## Test totale

In [ ]:
npzfile = np.load('system_matrices.npz')
A, B = npzfile['arr_0'], npzfile['arr_1']
A_B=np.concatenate( (A, B), axis=1).transpose(2, 0, 1)
C = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
C = np.stack([C, np.zeros((2, 4)), np.zeros((2, 4)), np.zeros((2, 4))], axis=2)
W = Polytope(
                A=np.block([[np.eye(4)], [-np.eye(4)]]), b=0.001 * np.ones((8, 1))
            )

E = Polytope(
                A=np.vstack((np.eye(2), -np.eye(2))),
                b=np.concatenate((np.ones(2) * 0.01, np.ones(2) * 0.01)),
            )

Theta_c = Polytope()

max_p, min_p = ([np.float64(-0.23350), np.float64(1.37460), np.float64(0.31949)], [np.float64(-1.81833),np.float64(-0.26129),np.float64(-0.61740)])

Theta = Polytope(A=np.block([[np.eye(3)], [-np.eye(3)]]), b=np.block([np.array(max_p), -np.array(min_p)]))  # np.block([np.array(max_p), -np.array(min_p)])

Q, R = np.eye(4), np.eye(2) #np.diag([1/(0.46**2), 1/(1.90**2)])
K = np.array(
    [
        [-7.57855882e-03, -6.40593493e-01, -1.83721441e-01, 5.62371887e-01],
        [-5.07279439e-05,  8.12719728e-02,  7.37019877e-02, 6.28097473e-02
         ]
        ]
    )

opt = {
    'K': K,
    "solver": 'osqp',
    "verbose": False,
    "svd": False,
    "xBound": (np.array([-1e4,-0.46, -0.01, -1.90]), np.array([1e4, 0.46, 0.01, 1.90])),
    "uBound": (np.array([-0.46, -1.90]), np.array([0.46, 1.90])),
    "name": 'tet_mpc',
    "W": W,
    'E': E,
    "theta": Theta,
    "lam": 0.98,
    'par_filter': 'lms',
    # 'ref': 'trajectory',
}

In [ ]:
controller = CRMPC({'A': A, 'B': B, 'C': C},
                         Q, R, 10, opt)
controller.initialize("LQR")
controller.P = np.array(
    [
        [ 1.90584932e+04, -3.11708260e+04, -2.18046990e+04, -3.82077431e+05],
        [-3.11708260e+04,  1.35038780e+07,  1.21994189e+07, -1.29229345e+07],
        [-2.18046990e+04,  1.21994189e+07,  1.24390961e+07, -1.31640429e+07],
        [-3.82077431e+05, -1.29229345e+07, -1.31640429e+07, 5.73129788e+07]
    ]
)

In [ ]:
controller.solve([0.3, 0.5, 0.01, 0.1])
controller.u_star

## Create a circle trajectory

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

delta_theta = [0]
speed_x = [0]

for i in range(1, 100):
    delta_theta.append(delta_theta[-1] - 2*np.pi*i/100)
    speed_x.append(np.cos(delta_theta[-1]))

# Export to csv file
import pandas as pd
df = pd.DataFrame({'delta_theta': delta_theta, 'speed_x': speed_x})
df.to_csv('trajectory_circle.csv', index=False)

plt.plot(np.rad2deg(delta_theta))
plt.plot(speed_x)

In [ ]:
import numpy as np
npzread = np.load('/home/stream/Personals/Fabio/ros2_ws/src/cRAMPC/config/system_matrices.npz')
A, B = npzread['arr_0'], npzread['arr_1']
A

In [ ]:

import csv

x, y, theta = [], [], []

with open('trajectory_circle_pose.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        x.append(float(row['x']))
        y.append(float(row['y']))
        theta.append(float(row['theta']))



[-1.5707963267948966, -1.5603243512829306, -1.5498523757709646, -1.5393804002589986, -1.5289084247470326, -1.5184364492350666, -1.5079644737231006, -1.4974924982111348, -1.4870205226991688, -1.4765485471872029, -1.4660765716752369, -1.4556045961632709, -1.4451326206513049, -1.4346606451393389, -1.424188669627373, -1.413716694115407, -1.403244718603441, -1.392772743091475, -1.382300767579509, -1.371828792067543, -1.361356816555577, -1.350884841043611, -1.3404128655316452, -1.3299408900196792, -1.3194689145077132, -1.3089969389957472, -1.2985249634837812, -1.2880529879718152, -1.2775810124598492, -1.2671090369478832, -1.2566370614359172, -1.2461650859239513, -1.2356931104119853, -1.2252211349000193, -1.2147491593880533, -1.2042771838760875, -1.1938052083641213, -1.1833332328521555, -1.1728612573401895, -1.1623892818282235, -1.1519173063162575, -1.1414453308042916, -1.1309733552923256, -1.1205013797803596, -1.1100294042683936, -1.0995574287564276, -1.0890854532444616, -1.0786134777324956,